# Train CNN variants: orig / clean × 5 seeds

In-scope ablation only (`insnorm` is out). Architecture and hparams match the draft.

| Variant | Features | N |
|---|---|---:|
| `orig` | `training/features/training.npz` | 7,200 |
| `clean` | `training/features/training-clean.npz` | 7,172 |

Seeds 42–46. Each run is an isolated OS process (`run_one.py`) so GPU memory is released.

**Split:** explicit stratified 80/10/10 (not Keras `validation_split`). Membership is in `results/splits/{variant}-seed{s}.csv` (`source_index`, `orig_index`, `file`, `label`, `split`). All 10 checkpoints are saved under `weights/`.

## Config

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import display, Markdown

HERE = Path.cwd()
if HERE.name != "cnn-latest":
    HERE = Path("training/notebooks/cnn-latest").resolve()
TRAINING_ROOT = HERE.parents[1]
PROGRESS = HERE / "results" / "progress.csv"
SEEDS = [42, 43, 44, 45, 46]
VARIANTS = ["orig", "clean"]

cfg = pd.DataFrame([
    {"key": "here", "value": str(HERE)},
    {"key": "orig", "value": str(TRAINING_ROOT / "features" / "training.npz")},
    {"key": "clean", "value": str(TRAINING_ROOT / "features" / "training-clean.npz")},
    {"key": "runs", "value": f"{len(VARIANTS)} x {len(SEEDS)} = {len(VARIANTS)*len(SEEDS)}"},
    {"key": "split", "value": "80/10/10 stratified, validation_data"},
])
display(cfg)
assert (TRAINING_ROOT / "features" / "training-clean.npz").is_file()
assert (HERE / "run_one.py").is_file()

## Training loop — one subprocess per (variant, seed)

In [ ]:
def load_progress():
    if PROGRESS.exists():
        return pd.read_csv(PROGRESS)
    return pd.DataFrame(columns=["variant", "seed"])

jobs = [(v, s) for v in VARIANTS for s in SEEDS]
done = load_progress()
pending = [(v, s) for v, s in jobs if done.empty or not ((done.variant == v) & (done.seed == s)).any()]
display(Markdown(f"Pending **{len(pending)}** / {len(jobs)} (already done: {len(jobs)-len(pending)})"))

for variant, seed in pending:
    display(Markdown(f"### `{variant}` seed {seed}"))
    result = subprocess.run(
        ["uv", "run", "python", "run_one.py", "--variant", variant, "--seed", str(seed)],
        cwd=HERE,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"{variant}/seed{seed} exited {result.returncode}")

progress = load_progress()
display(Markdown("### In-domain held-out test"))
display(progress.sort_values(["variant", "seed"]).reset_index(drop=True))

## Split sizes

In [ ]:
split_dir = HERE / "results" / "splits"
rows = []
for p in sorted(split_dir.glob("*-seed*.csv")):
    s = pd.read_csv(p)
    counts = s["split"].value_counts()
    n = len(s)
    rows.append({
        "file": p.name,
        "n": n,
        "train": int(counts.get("train", 0)),
        "val": int(counts.get("val", 0)),
        "test": int(counts.get("test", 0)),
        "train_frac": counts.get("train", 0) / n,
        "val_frac": counts.get("val", 0) / n,
        "test_frac": counts.get("test", 0) / n,
    })
if rows:
    display(pd.DataFrame(rows).round(4))
else:
    display(Markdown("No split CSVs yet."))